# Significance of ENS/GNNM improvements (cluster-level test)

The 10 official splits within a (backbone, dataset) cell are correlated (same graph), so pooling all 350 pairs would be pseudoreplication. Instead we **average the 10 splits to one score per cell** (35 independent cells), form paired differences, and test them with the **standardized Wilcoxon** (per-cell difference divided by its across-split std), one-sided `>0`. Selection per split mirrors `check_from_csv.ipynb` / `make_tables.py`: pick the max-`val_metric` row, read its test score (`test_roc_auc` for minesweeper/questions/tolokers, else `test_acc`).

In [1]:
from pathlib import Path
import numpy as np, pandas as pd
from scipy.stats import wilcoxon

CSV_ROOTS = [Path('results'), Path('results2'), Path('results3')]
DATASETS = ['roman_empire', 'amazon_ratings', 'minesweeper', 'questions', 'tolokers']
ROC_PRIMARY = {'questions', 'minesweeper', 'tolokers'}
MODELS = ['GCN', 'SAGE', 'GAT', 'GAT-sep', 'GT', 'GT-sep', 'ResNet']
VARIANT_FILES = {'BASE': 'base.csv', 'ENS': 'ensemble.csv', 'TABM': 'tabm.csv'}
NUM_SPLITS = 10

def read_csv_robust(path):
    try:
        first = path.open().readline().strip()
        if not first:
            return None
        if first.split(',', 1)[0] == 'dataset':
            df = pd.read_csv(path, skip_blank_lines=True)
        else:
            raw = pd.read_csv(path, header=None, skip_blank_lines=True)
            mask = raw.iloc[:, 0].astype(str) == 'dataset'
            if not mask.any():
                return None
            hi = int(mask.idxmax())
            df = raw.drop(index=hi).reset_index(drop=True)
            df.columns = raw.iloc[hi].astype(str).tolist()
            df = df[df.iloc[:, 0].astype(str) != 'dataset']
    except pd.errors.EmptyDataError:
        return None
    return None if df.empty else df[df['dataset'].astype(str) != 'dataset'].copy()

def load_variant(dataset, variant):
    frames = [d for r in CSV_ROOTS
              if (p := r / dataset / VARIANT_FILES[variant]).is_file()
              and (d := read_csv_robust(p)) is not None and not d.empty]
    if not frames:
        return None
    df = pd.concat(frames, ignore_index=True)
    df['val_metric'] = pd.to_numeric(df['val_metric'], errors='coerce')
    return df.dropna(subset=['val_metric'])

def best_test_per_split(df, model, dataset):
    col = 'test_roc_auc' if dataset in ROC_PRIMARY else 'test_acc'
    sub = df[df['model'] == model].copy()
    for c in ('split', 'val_metric', col):
        sub[c] = pd.to_numeric(sub[c], errors='coerce')
    sub = sub.dropna(subset=['split'])
    out = {}
    for split, grp in sub.groupby('split'):
        split = int(split)
        if 0 <= split < NUM_SPLITS and not (grp := grp.dropna(subset=['val_metric', col])).empty:
            out[split] = float(grp.loc[grp['val_metric'].idxmax(), col])
    return out

scores = {}
for dataset in DATASETS:
    for variant in VARIANT_FILES:
        df = load_variant(dataset, variant)
        if df is not None:
            for model in MODELS:
                if (s := best_test_per_split(df, model, dataset)):
                    scores[(dataset, model, variant)] = s

# One score per cell = mean over the splits common to BASE/ENS/TABM.
rows = []
for dataset in DATASETS:
    for model in MODELS:
        keys = [(dataset, model, v) for v in VARIANT_FILES]
        if not all(k in scores for k in keys):
            continue
        common = sorted(set.intersection(*(set(scores[k]) for k in keys)))
        if not common:
            continue
        get = lambda v: np.array([scores[(dataset, model, v)][s] for s in common])
        b, e, t = get('BASE'), get('ENS'), get('TABM')
        rows.append(dict(dataset=dataset, model=model, n=len(common),
                         ens_base=(e - b).mean(), tabm_ens=(t - e).mean(), tabm_base=(t - b).mean(),
                         sd_ens_base=(e - b).std(ddof=1), sd_tabm_ens=(t - e).std(ddof=1), sd_tabm_base=(t - b).std(ddof=1)))
cells = pd.DataFrame(rows)
print(f'{len(cells)} cells (backbone x dataset)')
cells[['dataset', 'model', 'n', 'ens_base', 'tabm_ens', 'tabm_base']].round(4)

35 cells (backbone x dataset)


,dataset,model,n,ens_base,tabm_ens,tabm_base
0,roman_empire,GCN,10,0.0189,0.0249,0.0438
1,roman_empire,SAGE,10,0.0078,0.0258,0.0336
2,roman_empire,GAT,10,0.0111,-0.0118,-0.0007
3,roman_empire,GAT-sep,10,0.0091,0.0105,0.0196
4,roman_empire,GT,10,0.0076,0.0124,0.0200
5,roman_empire,GT-sep,10,0.0075,0.0160,0.0235
6,roman_empire,ResNet,10,0.0029,-0.0015,0.0014
7,amazon_ratings,GCN,10,0.0037,-0.0070,-0.0033
8,amazon_ratings,SAGE,10,0.0115,0.0048,0.0163
9,amazon_ratings,GAT,10,0.0131,-0.0013,0.0118


In [2]:
def cluster_test(diff, sd, name):
    diff, sd = np.asarray(diff, float), np.asarray(sd, float)
    d = diff[np.isfinite(diff)]
    pos, neg = int((d > 0).sum()), int((d < 0).sum())
    std_d = (diff / sd); std_d = std_d[np.isfinite(std_d)]
    _, wil_std = wilcoxon(std_d, alternative='greater', zero_method='wilcox', method='exact')  # PRIMARY
    return dict(comparison=name, n=len(d), improved=pos, worse=neg,
                mean_pp=round(d.mean() * 100, 3), median_pp=round(float(np.median(d)) * 100, 3),
                wilcoxon_std_p=wil_std)

res = pd.DataFrame([
    cluster_test(cells['ens_base'],  cells['sd_ens_base'],  'ENS - BASE'),
    cluster_test(cells['tabm_ens'],  cells['sd_tabm_ens'],  'GNNM - ENS'),
    cluster_test(cells['tabm_base'], cells['sd_tabm_base'], 'GNNM - BASE'),
])
pd.set_option('display.float_format', lambda x: f'{x:.3g}')
res

,comparison,n,improved,worse,mean_pp,median_pp,wilcoxon_std_p
0,ENS - BASE,35,34,1,0.657,0.511,5.82e-11
1,GNNM - ENS,35,24,11,0.406,0.423,0.0083
2,GNNM - BASE,35,28,7,1.06,1.17,4.84e-06
